In [1]:
!pip install -q unsloth pyngrok
# !pip install "numpy<2" --force-reinstall

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.3/70.3 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 447.0/447.0 kB 11.1 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 29.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 28.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.3/188.3 MB 9.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 24.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 393.8/393.8 kB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 72.7 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 915.7/915.7 MB 2.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 18.6 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.3/594.3 MB 2.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB

In [ ]:
%%writefile run_api.py
import os
import uvicorn
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from pyngrok import ngrok
from unsloth import FastLanguageModel
from contextlib import asynccontextmanager
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()

# Global Variables
model = None
tokenizer = None

# Modern Lifespan Manager (Replaces @app.on_event)
@asynccontextmanager
async def lifespan(app: FastAPI):
    # --- Startup Logic ---
    global model, tokenizer
    print("Loading Custom Manga Translator Model...")
    save_directory = "/kaggle/input/models/pritepak/mt-jp-to-thai/other/default/1/Qwen3-14B-Base-unsloth-bnb-4bit"
    
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=save_directory,
        max_seq_length=2048,
        load_in_4bit=True,
    )
    FastLanguageModel.for_inference(model)
    print("Model loaded and ready for inference!")
    
    yield # The server runs while paused here
    
    # --- Shutdown Logic ---
    print("Shutting down server and clearing memory...")

app = FastAPI(title="Manga Translator API", lifespan=lifespan)

class TranslationRequest(BaseModel):
    japanese_text: str

@app.post("/translate")
async def translate_text(request: TranslationRequest):
    if not model or not tokenizer:
        raise HTTPException(status_code=503, detail="Model loading.")
    
    try:
        messages = [
            {"role": "system", "content": "You are an expert manga translator. Translate the Japanese text into natural, fluent Thai. Ensure the translation captures the original tone, emotion, and character nuances."},
            {"role": "user", "content": f"Japanese: {request.japanese_text}"}
        ]
        
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(text, return_tensors="pt").to("cuda")
        
        outputs = model.generate(**inputs, max_new_tokens=256, temperature=0.7, top_p=0.8, top_k=20)
        
        prompt_length = inputs['input_ids'].shape[1]
        pred_text = tokenizer.decode(outputs[0][prompt_length:], skip_special_tokens=True)
        
        return {"translated_text": pred_text}
        
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

if __name__ == "__main__":
    ngrok.kill()
    
    NGROK_TOKEN = user_secrets.get_secret("NGROK_AUTHTOKEN")
    ngrok.set_auth_token(NGROK_TOKEN)
    
    port = 8000
    public_url = ngrok.connect(port)
    print("=" * 60)
    print(f"🚀 FASTAPI RUNNING PUBLICLY AT: {public_url.public_url}")
    print("=" * 60)
    
    uvicorn.run(app, host="0.0.0.0", port=port)

In [ ]:
!python run_api.py

In [ ]:
# # Kill the specific process ID
# try:
#     process.terminate()
#     process.wait()
#     print("Server stopped.")
# except:
#     print("Server process not found or already stopped.")
    
# # Force kill python if stuck (nuclear option)
# !pkill -f run_api.py